# 01 - Chest X-ray: EDA and Data Preparation


> **Academic prototype.** This notebook is part of a university final project.
> The models here are **not** medical devices, are **not** validated on clinical
> data, and must **never** be used to diagnose, screen or triage real patients.
> See `docs/ETHICS.md`.


**Goal:** turn the raw X-ray folder into one validated manifest with a
**patient-level** train/validation/test split, and understand the data before
modelling.

Steps: config -> build manifest -> integrity checks -> class distribution ->
image sizes -> sample images -> patient-level split -> leakage check -> save.

**Before running:** point `data.root` and `data.labels` in
`configs/xray_baseline.yaml` at your dataset (see `data/README.md`).

In [ ]:
# Make `src/` importable no matter where Jupyter was launched from.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd

from src.common import ensure_dir, load_config, seed_everything
from src.common.io_utils import save_csv, save_json, save_figure
from src.common.viz import plot_class_distribution, plot_image_grid, set_plot_style
from src.classification import (
    assert_no_patient_leakage, build_manifest, check_manifest,
    image_size_report, patient_level_split, split_summary, use_existing_split,
)

set_plot_style()
pd.set_option("display.width", 140)

cfg = load_config("xray_baseline.yaml")
seed_everything(cfg.get("seed", 42), deterministic=cfg.get("deterministic", True))

LABELS = list(cfg.data.labels)
FIG_DIR = ensure_dir(PROJECT_ROOT / "outputs/classification/eda/figures")
MET_DIR = ensure_dir(PROJECT_ROOT / "outputs/classification/eda/metrics")

print("Dataset root :", cfg.path("data.root"))
print("Layout       :", cfg.get("data.layout"))
print("Labels       :", LABELS)
print("Image size   :", cfg.get("data.image_size"))
print("Seed         :", cfg.get("seed"))

## 1. Build the manifest

One tidy table describing every image: path, patient ID and one 0/1 column per label. If this cell raises, the error message names the exact path or column to fix in the config.

In [ ]:
manifest = build_manifest(cfg)
print(f"{len(manifest)} rows\n")
manifest.head()

## 2. Integrity checks

Missing files, duplicates, images with no label and the positive rate per label. Record these numbers in the report - they define what the metrics later actually mean.

In [ ]:
report = check_manifest(manifest, LABELS)
save_json(report, MET_DIR / "manifest_report.json")

In [ ]:
# Drop rows whose image file does not exist, so training cannot crash mid-epoch.
before = len(manifest)
manifest = manifest[manifest["image_path"].map(lambda p: Path(p).is_file())].reset_index(drop=True)
print(f"kept {len(manifest)} / {before} rows ({before - len(manifest)} missing files dropped)")

## 3. Class distribution

The imbalance seen here is exactly what `pos_weight` compensates for in the loss, and why accuracy is only a secondary metric.

In [ ]:
counts = {label: int(manifest[label].sum()) for label in LABELS}
counts["(no listed finding)"] = int((manifest[LABELS].sum(axis=1) == 0).sum())

fig = plot_class_distribution(counts, title="Chest X-ray: images per label (full dataset)")
save_figure(fig, FIG_DIR / "class_distribution.png", close=False)

for label in LABELS:
    rate = manifest[label].mean()
    print(f"{label:<24s} {counts[label]:>7d} positives  ({rate:.2%})  "
          f"-> a 'always negative' model would score {1 - rate:.2%} accuracy")

## 4. Images per patient

If patients have several images, a random per-image split would leak. This is the justification for the patient-level split in section 7.

In [ ]:
per_patient = manifest.groupby("patient_id").size()
print(per_patient.describe().round(2).to_string())
print(f"\nPatients with >1 image: {(per_patient > 1).sum()} / {len(per_patient)}")

if (per_patient > 1).any():
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(per_patient.values, bins=min(30, int(per_patient.max())), color="#4C72B0",
            edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Images per patient")
    ax.set_ylabel("Number of patients")
    ax.set_title("Images per patient")
    save_figure(fig, FIG_DIR / "images_per_patient.png", close=False)
else:
    print("\n[note] One image per patient (or no patient IDs available). "
          "State this in the report - if IDs were unavailable, the split is per-image "
          "and leakage cannot be ruled out.")

## 5. Image dimensions and formats

Mixed sizes and colour modes are normal in X-ray datasets; everything is resized to `data.image_size` and converted to 3-channel at load time. Corrupt files show up here as errors.

In [ ]:
sizes = image_size_report(manifest, sample=300, seed=cfg.get("seed", 42))
save_csv(sizes, MET_DIR / "image_size_sample.csv")

if "error" in sizes.columns and sizes["error"].notna().any():
    print("[warn] unreadable files found:")
    print(sizes[sizes["error"].notna()][["image_path", "error"]].head().to_string(index=False))

print(sizes[["width", "height"]].describe().round(1).to_string())
print("\nMost common (width x height):")
print(sizes.groupby(["width", "height"]).size().sort_values(ascending=False).head(5).to_string())
print("\nColour modes:", sizes["mode"].value_counts().to_dict())

## 6. Sample images

Look at the data before modelling it. Watch for: burnt-in text markers, rotated or lateral views, and inverted greyscale - all of these are shortcuts a CNN will happily learn instead of the pathology.

In [ ]:
from PIL import Image

rng = np.random.default_rng(cfg.get("seed", 42))
label = LABELS[0]

for value, title in [(1, f"{label} = 1 (positive)"), (0, f"{label} = 0 (negative)")]:
    subset = manifest[manifest[label] == value]
    if len(subset) == 0:
        print(f"[warn] no examples with {title}")
        continue
    picks = subset.iloc[rng.choice(len(subset), size=min(4, len(subset)), replace=False)]
    images = [np.array(Image.open(p).convert("L")) for p in picks["image_path"]]
    titles = [f"{Path(p).name}\n{h}x{w}" for p, (h, w) in
              zip(picks["image_path"], [im.shape for im in images])]
    fig = plot_image_grid(images, titles, ncols=4, suptitle=f"Samples: {title}")
    save_figure(fig, FIG_DIR / f"samples_{label}_{value}.png", close=False)

## 7. Patient-level split

**This is the methodological core of the notebook.** Splitting by patient rather
than by image is what stops near-identical images of the same person appearing in
both training and test.

`use_official_split: true` keeps the dataset's own train/test folders (so results
stay comparable with published work) and carves the validation set out of train.
Set it to `false` for a fresh three-way split.

In [ ]:
val_size = float(cfg.get("data.val_size", 0.15))
test_size = float(cfg.get("data.test_size", 0.15))
seed = int(cfg.get("seed", 42))
stratify_col = LABELS[0] if len(LABELS) >= 1 else None

if bool(cfg.get("data.use_official_split", True)) and "split" in manifest.columns:
    print("Using the dataset's official split; carving validation out of train.\n")
    split_df = use_existing_split(manifest, val_size=val_size, seed=seed, stratify_col=stratify_col)
else:
    print(f"Building a fresh patient-level split "
          f"(train={1 - val_size - test_size:.0%}, val={val_size:.0%}, test={test_size:.0%}).\n")
    split_df = patient_level_split(manifest.drop(columns=["split"], errors="ignore"),
                                   val_size=val_size, test_size=test_size, seed=seed,
                                   stratify_col=stratify_col)

split_df["split"].value_counts()

### 7a. Leakage check

This raises an exception if any patient appears in more than one split. **Screenshot this output for the report** - it is the evidence that the evaluation is sound.

In [ ]:
leakage = assert_no_patient_leakage(split_df)
save_json(leakage, MET_DIR / "split_leakage_check.json")

### 7b. Split summary

The positive rate should be similar across splits. A test set with a very different prevalence makes the test metrics hard to compare with validation.

In [ ]:
summary = split_summary(split_df, LABELS)
save_csv(summary, MET_DIR / "split_summary.csv")
summary

In [ ]:
fig = plot_class_distribution(
    {f"{s} ({int(g[LABELS[0]].sum())}+)": len(g) for s, g in split_df.groupby("split")},
    title=f"Images per split (positives for '{LABELS[0]}' in brackets)",
)
save_figure(fig, FIG_DIR / "split_distribution.png", close=False)

## 8. Save the prepared manifest

Notebooks 02 and 03 read this file, so the split is fixed once and reused - training and evaluation cannot silently disagree about which images are in the test set.

In [ ]:
MANIFEST_PATH = ensure_dir(PROJECT_ROOT / "data/processed") / "xray_manifest.csv"
save_csv(split_df, MANIFEST_PATH)

print("\nColumns:", list(split_df.columns))
print(f"\nNext: run 02_xray_training.ipynb (it loads {MANIFEST_PATH.name})")

---

## Findings to write up

Fill these in from the outputs above - do **not** guess:

- dataset size, number of patients, images per patient;
- class balance and what "always predict negative" would score;
- image size / format variation;
- any corrupt or missing files removed;
- split strategy and the leakage-check result.

### Screenshots for the report
- Section 3 class-distribution figure
- Section 6 sample-image grids
- Section 7a leakage check output
- Section 7b split summary table